# PT-Flow — CIFAR-10, full run (relay training across sessions)

Class-conditional CIFAR-10, all 10 classes, all 50,000 images, native 32×32 pixel space, conv
backbones. One-step (1-NFE) sampling. Designed to be run as a **relay**: each session trains for
~11 h, checkpoints, and the next session (possibly on a different account) picks up exactly where it
left off, with the ε-anneal state and a single continuous W&B curve.

## The budget, honestly

Per training sample this costs ~95 GFLOP (`K=32`, `eta_steps=2`, generator `ch=96`). A 2×T4 session
sustains roughly 6.5 TFLOPS in fp32, so:

| | |
|---|---|
| One 11 h session | ~2.7M samples = **~10,500 steps at batch 256** |
| Target: 800,000 steps × 256 | **205M samples** (EDM / iCT-class budget) |
| Sessions needed | **~76** |
| With 7 people relaying (~2.5 sessions/week each) | **~4–5 weeks** |

**Why batch 256 and not 1024.** Your 800,000-step figure is right — at batch **256**. 800k × 1024
would be 819M samples, ~8× DDPM's entire budget and ~93 days of continuous 2×T4 compute. At batch
256, 800k steps lands on 205M samples, which is the budget EDM and iCT actually use for CIFAR-10.
Smaller batches also give more optimizer steps per sample, which matters more than raw batch size at
a fixed FLOP budget.

**The single biggest speedup available** is fp16: T4 has fp16 tensor cores (65 TFLOPS vs 8.1 fp32),
and the codebase currently runs pure fp32 because Turing has no bf16. Adding fp16 AMP + `GradScaler`
would cut the 4–5 weeks to roughly **2 weeks**. It is not implemented yet.

> **Sidebar:** Internet **ON**, Accelerator **GPU T4 × 2**, secrets `GH_TOKEN` (GitHub read) and
> `WANDB_API_KEY`. To continue a run, also attach the previous session's output as a **Data source**.


## 1 — Setup, clone, W&B

For the relay to show up as **one** curve, every session must use the same `WANDB_RUN_ID` and the
same W&B project — and the 7 accounts must share one W&B API key (or one W&B team).


In [1]:
import os, sys, json, glob, time, shutil, subprocess

WORK      = "/kaggle/working"
REPO      = f"{WORK}/PT-FLow"
DATA_DIR  = f"{WORK}/ptflow_data/cifar10"
FID_NPZ   = f"{WORK}/cifar10_train_fid_stats.npz"
TORCH_HUB = f"{WORK}/torch_hub"
WORKDIR   = f"{WORK}/runs/ptflow_cifar10"
for d in (DATA_DIR, TORCH_HUB, os.path.dirname(WORKDIR)):
    os.makedirs(d, exist_ok=True)

# ===================== RUN-DEFINING KNOBS =====================
# These define the SCHEDULE and must be IDENTICAL in every relay session.
TARGET_STEPS = 800_000     # nominal schedule length (205M samples at B=256)
BATCH_SIZE   = 256         # global; 128/rank on 2 GPUs
K_PROPOSALS  = 32          # estimator budget (dominant cost term)
GEN_CH       = 96          # generator: 64 -> 7.1M, 96 -> 15.0M, 128 -> 25.8M
POT_CH       = 64          # potential: encoder->scalar, called K+1x per sample
EPS_FINAL    = 1e-3
EMA_DECAY    = 0.9999      # ~10k-step horizon, right for a long run
WANDB_PROJECT = "ptflow-cifar10"
WANDB_RUN_ID  = "cifar10-b256-k32-g96-v1"   # <-- same string in EVERY session
# ===================== PER-SESSION KNOBS ======================
SESSION_HOURS = 8.5        # training budget THIS session (Kaggle caps ~12h)
EVAL_HOURS    = 2.0        # reserved for the evaluation suite below
RUN_EVAL      = True       # set False on intermediate relay legs to train longer
FID_SWEEP_N   = 10_000     # samples per guidance weight in the sweep
FID_FINAL_N   = 50_000     # headline FID at the best w
GITHUB_REPO   = "github.com/kraihan/PT-FLow.git"
# =============================================================

if not os.path.isdir(REPO):
    from kaggle_secrets import UserSecretsClient
    try:
        _tok = UserSecretsClient().get_secret("GH_TOKEN")
    except Exception as e:
        raise RuntimeError("Add a Kaggle Secret GH_TOKEN with a GitHub read token.") from e
    r = subprocess.run(["git", "clone", f"https://{_tok}@{GITHUB_REPO}", REPO],
                       capture_output=True, text=True)
    del _tok
    if r.returncode != 0:
        raise RuntimeError("clone failed: " + r.stderr[:400])
    print("cloned")
else:
    print("repo already present")

get_ipython().system("pip install -q torch-fidelity einops absl-py wandb")

import torch
NGPU = max(1, torch.cuda.device_count())
print("torch", torch.__version__, "| GPUs", NGPU,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

# ---- W&B: one shared run across every relay session ----
USE_WANDB = True
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print(f"W&B enabled -> project '{WANDB_PROJECT}', run id '{WANDB_RUN_ID}'")
except Exception as e:
    USE_WANDB = False
    print("W&B disabled (no WANDB_API_KEY secret); metrics go to the local jsonl instead")

os.environ.update(
    CIFAR10_PATH=DATA_DIR, CIFAR10_FID_NPZ=FID_NPZ, TORCH_HUB_DIR=TORCH_HUB,
    PYTHONPATH=REPO, OMP_NUM_THREADS="2", TOKENIZERS_PARALLELISM="false",
)
if REPO not in sys.path:
    sys.path.insert(0, REPO)


cloned
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 2.8 MB/s eta 0:00:00
torch 2.10.0+cu128 | GPUs 2 | Tesla T4
W&B disabled (no WANDB_API_KEY secret); metrics go to the local jsonl instead


In [2]:
# ============ proposal numbers: R0 reversal + exact-identity residuals ============
# Pure closed-form / small-MC on hand-designed potentials. No training, no data.
import math, json, sys
import numpy as np, torch

if REPO not in sys.path: sys.path.insert(0, REPO)
from ptflow.core.estimator import log_psi0_tilted, sample_proposal
from ptflow.core.likelihood import log_likelihood

DEV  = "cuda" if torch.cuda.is_available() else "cpu"
DIMS = [2, 16, 64, 128]        # Tier 3 comes free here
K, B, A_Q, CUB = 512, 64, 0.6, 0.05
EPS = np.logspace(-4, 2, 25)   # spans warm enough that the curves actually cross
print(f"device {DEV} | phi = {A_Q}/2|y|^2 + {CUB}*sum y_i^3   (M3 = {6*CUB:g})\n")

# ---- potential with an exact Newton prox and a genuine third derivative ----
dphi  = lambda y: A_Q*y + 3*CUB*y.pow(2)
d2phi = lambda y: A_Q + 6*CUB*y
def prox(x, iters=80):
    y = x.clone()
    for _ in range(iters): y = y - (dphi(y) + y - x) / (d2phi(y) + 1.0)
    return y
mk_u = lambda eps: (lambda y: (0.5*A_Q*y.pow(2).sum(-1) + CUB*y.pow(3).sum(-1)) / (2*eps))

rows = []
for d in DIMS:
    torch.manual_seed(0)
    for eps in EPS:
        eps = float(eps); u = mk_u(eps)
        x0 = 0.5*torch.randn(B, d, device=DEV)
        m  = prox(x0); s = -torch.log1p(d2phi(m).clamp_min(1e-3))
        pr = sample_proposal(m, s, x0, eps=eps, K=K, alpha_def=0.0, beta_data=0.0)
        s2t = float(log_psi0_tilted(u, pr, x0, eps).s2_res)
        z   = torch.randn(K, B, d, device=DEV)
        s2n = float((-u((x0[None] + math.sqrt(2*eps)*z).reshape(-1, d)).reshape(K, B))
                    .var(dim=0, unbiased=False).mean())
        rows.append({"d": d, "eps": eps, "s2_tilted": s2t, "s2_naive": s2n})

def slope(x, y):   # d log10(s2) / d log10(eps)
    return float(np.polyfit(np.log10(x), np.log10(np.maximum(y, 1e-300)), 1)[0])

print("="*80); print("A. R0 — VARIANCE REVERSAL"); print("="*80)
summary = {}
for d in DIMS:
    rd = sorted([r for r in rows if r["d"] == d], key=lambda z: z["eps"])
    e  = [r["eps"] for r in rd]; st = [r["s2_tilted"] for r in rd]; sn = [r["s2_naive"] for r in rd]
    diff = np.log10(st) - np.log10(sn); cross = None
    for i in range(len(diff)-1):
        if diff[i] < 0 <= diff[i+1]:
            t = diff[i]/(diff[i]-diff[i+1])
            cross = 10**(np.log10(e[i]) + t*(np.log10(e[i+1])-np.log10(e[i]))); break
    summary[d] = {"slope_tilted": slope(e, st), "slope_naive": slope(e, sn),
                  "crossing_eps": cross, "decades": math.log10(max(e)/min(e))}
    S = summary[d]
    print(f"\nd = {d}   slopes: tilted {S['slope_tilted']:+.3f} (theory +1)   "
          f"naive {S['slope_naive']:+.3f} (theory -1)")
    print(f"   crossing eps = {f'{cross:.3g}' if cross else 'none in range'}   "
          f"| {S['decades']:.0f} orders of magnitude covered")
    print(f"   {'eps':>9} {'s2 tilted':>12} {'s2 naive':>12} {'K tilted':>11} {'K naive':>13}")
    for r in rd:
        if not any(abs(math.log10(r["eps"])-t) < 1e-6 for t in (-4,-3,-2,-1,0,1,2)): continue
        print(f"   {r['eps']:>9.1e} {r['s2_tilted']:>12.3e} {r['s2_naive']:>12.3e} "
              f"{10**min(r['s2_tilted']/math.log(10),300):>11.2e} "
              f"{10**min(r['s2_naive']/math.log(10),300):>13.2e}")

# ---- B. exact identities on the quadratic fixture (remainder identically 0) ----
print("\n" + "="*80); print("B. EXACT-IDENTITY RESIDUALS"); print("="*80)
a, eps, d = 0.7, 0.01, 8
u_q  = lambda y: 0.5*a*y.pow(2).sum(-1)/(2*eps)
ex   = lambda x: -0.5*d*math.log(1+a) - a*x.pow(2).sum(-1)/(4*eps*(1+a))
torch.manual_seed(0)
x0 = torch.randn(64, d, device=DEV); m = x0/(1+a)
pr = sample_proposal(m, torch.full_like(x0, -math.log(1+a)), x0, eps=eps, K=64,
                     alpha_def=0.0, beta_data=0.0)
est = log_psi0_tilted(u_q, pr, x0, eps)
rel = float(((est.log_psi0-ex(x0)).abs()/ex(x0).abs().clamp_min(1)).max())
mt  = float((x0 - a*x0/(1+a) - x0/(1+a)).abs().max())
print(f"\n  closed-form psi_0 (quadratic phi):  max rel error {rel:.3e}"
      f"   ESS/K {float(est.ess_frac):.6f}   s2_res {float(est.s2_res):.3e}")
print(f"  Moreau+Tweedie  T(x0)=x0-grad phi_0=prox(x0):  max abs error {mt:.3e}")

# ---- normalization by grid quadrature, along the IWAE ladder ----
a2, eps2, d2_, lim, n = 0.6, 0.05, 2, 5.0, 61
uf  = lambda x, c: 0.5*a2*x.pow(2).sum(-1)/(2*eps2)
gf  = lambda x0, c, w=0.0: (x0/(1+a2), torch.full_like(x0, -math.log(1+a2)))
g   = torch.linspace(-lim, lim, n, device=DEV)
mesh= torch.stack(torch.meshgrid(*([g]*d2_), indexing="ij"), -1).reshape(-1, d2_)
cz  = torch.zeros(mesh.shape[0], dtype=torch.long, device=DEV)
cell= (2*lim/(n-1))**d2_
print(f"\n  normalization  int psi_hat_1 psi_1 dx = 1   (Thm 12.1)")
print(f"    grid d={d2_}, [-{lim},{lim}]^{d2_}, {n}^{d2_} pts, eps={eps2}")
print(f"    {'K_eval':>8} {'mass':>12} {'|mass-1|':>12}")
ladder = []
for Ke in (16, 64, 256):
    torch.manual_seed(0)
    mass = float(torch.exp(log_likelihood(uf, gf, mesh, cz, eps2,
                 K_inner=Ke, M_outer=Ke, alpha_def=0.0)).sum()*cell)
    ladder.append({"K_eval": Ke, "mass": mass, "abs_error": abs(mass-1)})
    print(f"    {Ke:>8} {mass:>12.6f} {abs(mass-1):>12.3e}")
print("    (IWAE LOWER bound => mass rises toward 1 with K)")

json.dump({"r0": rows, "summary": {str(k): v for k, v in summary.items()},
           "psi0_rel_err": rel, "moreau_abs_err": mt, "normalization": ladder},
          open(f"{WORK}/proposal_numbers.json", "w"), indent=2)
print(f"\nsaved {WORK}/proposal_numbers.json")

device cuda | phi = 0.6/2|y|^2 + 0.05*sum y_i^3   (M3 = 0.3)

A. R0 — VARIANCE REVERSAL

d = 2   slopes: tilted +1.003 (theory +1)   naive -0.382 (theory -1)
   crossing eps = none in range   | 6 orders of magnitude covered
         eps    s2 tilted     s2 naive    K tilted       K naive
     1.0e-04    3.551e-06    8.298e+02    1.00e+00     1.00e+300
     1.0e-03    3.730e-05    8.492e+01    1.00e+00      7.58e+36
     1.0e-02    3.805e-04    1.205e+01    1.00e+00      1.71e+05
     1.0e-01    3.745e-03    1.170e+00    1.00e+00      3.22e+00
     1.0e+00    3.981e-02    6.122e-01    1.04e+00      1.84e+00
     1.0e+01    4.111e-01    1.926e+00    1.51e+00      6.86e+00
     1.0e+02    3.887e+00    1.514e+01    4.88e+01      3.75e+06

d = 16   slopes: tilted +0.999 (theory +1)   naive -0.382 (theory -1)
   crossing eps = none in range   | 6 orders of magnitude covered
         eps    s2 tilted     s2 naive    K tilted       K naive
     1.0e-04    2.985e-05    7.076e+03    1.00e+00    

In [3]:
COLD = 1e-2
print(f"\n{'d':>5} {'tilted (all)':>13} {'naive (cold branch)':>21}")
for d in DIMS:
    rd = sorted([r for r in rows if r["d"] == d], key=lambda z: z["eps"])
    c  = [r for r in rd if r["eps"] <= COLD]
    print(f"{d:>5} {slope([r['eps'] for r in rd], [r['s2_tilted'] for r in rd]):>+13.3f}"
          f" {slope([r['eps'] for r in c], [r['s2_naive'] for r in c]):>+21.3f}")
    print(f"      s2_tilted/(d*eps) = "
          f"{np.mean([r['s2_tilted']/(d*r['eps']) for r in rd]):.4f}  (should be constant)")


    d  tilted (all)   naive (cold branch)
    2        +1.003                -0.955
      s2_tilted/(d*eps) = 0.0188  (should be constant)
   16        +0.999                -0.976
      s2_tilted/(d*eps) = 0.0187  (should be constant)
   64        +1.000                -0.992
      s2_tilted/(d*eps) = 0.0188  (should be constant)
  128        +0.999                -0.989
      s2_tilted/(d*eps) = 0.0188  (should be constant)


In [4]:
# ---- 2D PoC: train the 8-Gaussian ring, then count recovered modes ----
import math, torch, numpy as np, json
get_ipython().system(f"cd {REPO} && python train_toy.py --config configs/toy_gmm.yaml "
                     f"--workdir {WORK}/runs/toy --device cuda")

from ptflow.core.sampler import sample_mode_a
from ptflow.data.toy import ToyStream
from ptflow.models.adapters import make_gen_fn
from ptflow.models.builder import build_networks
from ptflow.utils.misc import load_config
import glob

cfg = load_config(f"{REPO}/configs/toy_gmm.yaml")
pot, gen, fs = build_networks(cfg)
ck = sorted(glob.glob(f"{WORK}/runs/toy/checkpoints/state_*.pt"))[-1]
pay = torch.load(ck, map_location="cpu", weights_only=False)
gen.load_state_dict(pay.get("ema_generator") or pay["generator"]); gen.eval()

stream = ToyStream("gaussian8", batch_size=1, dim=2)      # for mean/std
ang = 2*math.pi*torch.arange(8).float()/8
centres = (torch.stack([2.0*torch.cos(ang), 2.0*torch.sin(ang)], -1)
           - stream.mean) / stream.std                     # into normalized space
sigma = float((0.1/stream.std).mean())

N = 8192
x0 = torch.randn(N, 2)
c  = torch.randint(0, 8, (N,))
x  = sample_mode_a(make_gen_fn(gen, fs), x0, c, 0.0)       # strict 1-NFE

dist = torch.cdist(x, centres)                             # (N, 8)
near, who = dist.min(1)
counts = torch.bincount(who, minlength=8)
recovered = int(((counts.float()/N) >= 0.01).sum())        # >=1% of mass
within3 = float((near <= 3*sigma).float().mean())

print(f"\nmodes recovered : {recovered}/8   (>=1% of samples each)")
print(f"per-mode share  : {[f'{v:.3f}' for v in (counts.float()/N).tolist()]}")
print(f"expected uniform: 0.125   max deviation {float((counts.float()/N - 0.125).abs().max()):.3f}")
print(f"within 3-sigma  : {within3:.1%}   (sample quality, not just coverage)")
print(f"1-NFE, checkpoint {ck.split('/')[-1]}")

[    1.2s] step       0 | L_theta=-0.0081 L_phi=-0.0032 ess=0.942 eps=0.2000 res=0.0080 disp=0.000
[    3.5s] step     100 | L_theta=-0.0848 L_phi=-0.0339 ess=0.940 eps=0.1996 res=0.0063 disp=0.099
[    5.6s] step     200 | L_theta=-1.4463 L_phi=-0.5744 ess=0.345 eps=0.1986 res=1.4022 disp=0.373
[    8.2s] step     300 | L_theta=-1.8066 L_phi=-0.7134 ess=0.306 eps=0.1974 res=1.4337 disp=0.455
[   10.5s] step     400 | L_theta=-1.8474 L_phi=-0.7226 ess=0.309 eps=0.1956 res=1.1058 disp=0.454
[   12.8s] step     500 | L_theta=-1.8675 L_phi=-0.7208 ess=0.313 eps=0.1930 res=0.7623 disp=0.458
[   15.0s] step     600 | L_theta=-1.7951 L_phi=-0.6809 ess=0.317 eps=0.1897 res=0.6863 disp=0.450
[   17.2s] step     700 | L_theta=-1.8824 L_phi=-0.7004 ess=0.322 eps=0.1860 res=0.7773 disp=0.455
[   19.4s] step     800 | L_theta=-1.9382 L_phi=-0.7043 ess=0.319 eps=0.1817 res=0.5048 disp=0.510
[   21.7s] step     900 | L_theta=-1.8618 L_phi=-0.6589 ess=0.313 eps=0.1770 res=0.6737 disp=0.466
[   23.9s]

In [5]:
# ---- NLL: full IWAE ladder vs analytic ground truth (toy) ----
import math, glob, torch
from ptflow.core.likelihood import nll_ladder
from ptflow.core.schedule import AnnealConfig, AnnealController
from ptflow.data.toy import ToyStream
from ptflow.models.adapters import make_gen_fn, make_u_fn
from ptflow.models.builder import build_networks
from ptflow.utils.misc import load_config

cfg = load_config(f"{REPO}/configs/toy_gmm.yaml")
pot, gen, fs = build_networks(cfg)
ck  = sorted(glob.glob(f"{WORK}/runs/toy/checkpoints/state_*.pt"))[-1]
pay = torch.load(ck, map_location="cpu", weights_only=False)
pot.load_state_dict(pay.get("ema_potential") or pay["potential"]); pot.eval()
gen.load_state_dict(pay.get("ema_generator") or pay["generator"]); gen.eval()
an = AnnealController(AnnealConfig(**dict(cfg.get("anneal", {})))); an.load_state_dict(pay["anneal"])

stream = ToyStream("gaussian8", batch_size=256, dim=2)
x, c = stream.sample(256)
out = nll_ladder(make_u_fn(pot, fs), make_gen_fn(gen, fs), x, c, an.eps,
                 ladder=(16, 64, 256, 1024, 4096), point_chunk=8192)

sigma = float((0.1 / stream.std).mean())
H_true = math.log(8) + math.log(2 * math.pi * math.e * sigma**2)   # nats, 2-D
print(f"eps = {an.eps:.4g}   analytic NLL = {H_true:+.3f} nats "
      f"({H_true/2:+.3f}/dim)\n")
print(f"{'K_eval':>8} {'NLL (nats)':>12} {'gap to truth':>13}")
for K in (16, 64, 256, 1024, 4096):
    v = out[f"nll_K{K}"]
    print(f"{K:>8} {v:>12.3f} {v - H_true:>13.3f}")

KeyboardInterrupt: 

### 1b — Verify the clone

No source patching: this run uses the repo as published, on all 10 classes and all 50k images.


In [10]:
_need = ["ptflow/models/unet.py", "configs/cifar10_unet.yaml",
         "configs/cifar10_unet_smoke.yaml", "scripts/make_ref_stats.py"]
_missing = [f for f in _need if not os.path.exists(f"{REPO}/{f}")]
assert not _missing, f"missing from the clone: {_missing} -- push them, or delete {REPO} and re-clone"

_dit = open(f"{REPO}/ptflow/models/dit.py").read()
_dst = open(f"{REPO}/ptflow/utils/dist_util.py").read()
_trn = open(f"{REPO}/ptflow/train/trainer.py").read()
_bld = open(f"{REPO}/ptflow/models/builder.py").read()
checks = {
    "RoPE buffers not aliased": "copy=True" in _dit,
    "DDP broadcast_buffers disabled": ("broadcast_buffers=False" in _dst
                                        or '"broadcast_buffers": False' in _dst),
    "ESS all-reduced before control flow": "all_reduce_mean" in _dst and "all_reduce_mean" in _trn,
    "builder has kind:unet": "unet" in _bld,
}
for k, v in checks.items():
    print(f"  [{'OK ' if v else 'MISSING'}] {k}")
assert all(checks.values()), "clone is stale -- push the current repo and re-clone"
print("\nrepo is current")


  [OK ] RoPE buffers not aliased
  [OK ] DDP broadcast_buffers disabled
  [OK ] ESS all-reduced before control flow
  [OK ] builder has kind:unet

repo is current


## 2 — CIFAR-10 + the 50k FID reference

Standard protocol: score generated samples against Inception statistics of the **full 50,000-image
train split**, built with the repo's own TF-compatible Inception (the same one `inference.py
evaluate` uses — mixing extractors silently produces meaningless FID).

Cached in `/kaggle/working`, so on a relay leg with the previous output attached this is instant.


In [12]:
GDRIVE_ID = "1AMESa9etn7VmnXb3GPbO2DWYttULbRvY"
CIFAR_MD5 = "c58f30108f718f92721af3b95e74349a"

# reuse a reference carried in from a previous session if one is attached
if not os.path.exists(FID_NPZ):
    for cand in glob.glob("/kaggle/input/*/cifar10_train_fid_stats.npz"):
        shutil.copy(cand, FID_NPZ); print("reused FID reference from", cand); break

if not os.path.exists(f"{DATA_DIR}/cifar-10-batches-py/data_batch_1"):
    get_ipython().system("pip install -q gdown")
    get_ipython().system(f"gdown {GDRIVE_ID} -O {DATA_DIR}/cifar-10-python.tar.gz")
    import hashlib
    got = hashlib.md5(open(f"{DATA_DIR}/cifar-10-python.tar.gz", "rb").read()).hexdigest()
    assert got == CIFAR_MD5, f"gdown fetched the wrong file (md5 {got})"
    from torchvision.datasets import CIFAR10
    CIFAR10(root=DATA_DIR, train=True, download=True)
    CIFAR10(root=DATA_DIR, train=False, download=True)

if not os.path.exists(FID_NPZ):
    rc = os.system(f"cd {REPO} && python scripts/make_ref_stats.py --dataset cifar10 "
                   f"--out {FID_NPZ} --batch-size 250")
    assert rc == 0, "FID reference build failed"

import numpy as np
_z = np.load(FID_NPZ)
assert _z["mu"].shape == (2048,) and _z["sigma"].shape == (2048, 2048)
print("FID reference ready:", {k: _z[k].shape for k in _z.files})


FID reference ready: {'mu': (2048,), 'sigma': (2048, 2048)}


## 3 — Config

`anneal_steps` is pinned to `0.8 × TARGET_STEPS` and **never** to the per-session step count — that
is what keeps the ε schedule continuous across the relay. The LR schedule is `const`, so it is
session-independent by construction. Only `train.total_steps` varies per session (§5), acting as
"where this leg stops".


In [15]:
import yaml
from ptflow.utils.misc import load_config

cfg = json.loads(json.dumps(load_config(f"{REPO}/configs/cifar10_unet.yaml")))

cfg["logging"] = {"use_wandb": USE_WANDB, "log_every_k": 50,
                  "project": WANDB_PROJECT, "run_id": WANDB_RUN_ID}
cfg["dataset"]["num_classes"] = 10
cfg["dataset"]["batch_size"] = BATCH_SIZE
cfg["dataset"]["eval_batch_size"] = 250
cfg["dataset"]["kwargs"]["num_workers"] = 2
cfg["dataset"]["use_aug"] = True

cfg["model"]["generator"]["ch"] = GEN_CH
cfg["model"]["potential"]["ch"] = POT_CH

cfg["train"]["micro_batches"] = 2          # K*B_mb sequences must fit in 15GB
cfg["train"]["eta_steps"] = 2
cfg["train"]["ema_decay"] = EMA_DECAY
cfg["train"]["ema_warmup"] = 5000
cfg["train"]["diag_every"] = 5000          # prox_multistart = 80 sequential fwd+bwd
cfg["train"]["save_per_step"] = 2000       # survive a session kill
cfg["train"]["keep_last"] = 2
cfg["train"]["keep_every"] = 100000
cfg["train"]["eval_per_step"] = 10**9      # eval runs below, not inline
cfg["train"]["potential_loss"]["K"] = K_PROPOSALS
cfg["optimizer"]["warmup_steps"] = 5000
cfg["anneal"]["eps_final"] = EPS_FINAL
cfg["anneal"]["anneal_steps"] = int(0.8 * TARGET_STEPS)    # SCHEDULE-DEFINING, never per-session
cfg["eval"]["enabled"] = False

CONFIG = f"{REPO}/configs/cifar10_pro.yaml"

def write_config(stop_at_step, use_wandb=None):
    cfg["train"]["total_steps"] = int(stop_at_step)
    if use_wandb is not None:
        cfg["logging"]["use_wandb"] = bool(use_wandb)
    yaml.safe_dump(cfg, open(CONFIG, "w"), sort_keys=False)

write_config(TARGET_STEPS)

from ptflow.models.builder import build_networks
pot, gen, fs = build_networks(load_config(CONFIG))
n_p = sum(q.numel() for q in pot.parameters()); n_g = sum(q.numel() for q in gen.parameters())
print(f"potential {n_p/1e6:.2f}M | generator {n_g/1e6:.2f}M | d={fs.dim}")

from torch.utils.flop_counter import FlopCounterMode
_x = torch.randn(2, 32, 32, 3); _c = torch.zeros(2, dtype=torch.long)
_f = {}
for _n, _call in (("pot", lambda: pot(_x, _c)), ("gen", lambda: gen(_x, _c, 0.5))):
    _m = FlopCounterMode(display=False)
    with _m: _call()
    _f[_n] = _m.get_total_flops() / 2 / 1e9
PER_SAMPLE_GFLOP = (3*K_PROPOSALS + 3 + 2*4) * _f["pot"] + (1 + 2*3) * _f["gen"]
print(f"potential {_f['pot']:.2f} GFLOP/img | generator {_f['gen']:.2f} GFLOP/img")
print(f"=> {PER_SAMPLE_GFLOP:.0f} GFLOP per training sample "
      f"({100*(3*K_PROPOSALS+3+2*4)*_f['pot']/PER_SAMPLE_GFLOP:.0f}% potential)")
print(f"\nTARGET {TARGET_STEPS:,} steps x B={BATCH_SIZE} = "
      f"{TARGET_STEPS*BATCH_SIZE/1e6:.0f}M samples ({TARGET_STEPS*BATCH_SIZE/50000:.0f} epochs)")
del pot, gen


potential 1.66M | generator 14.99M | d=3072
potential 0.43 GFLOP/img | generator 7.02 GFLOP/img
=> 95 GFLOP per training sample (48% potential)

TARGET 800,000 steps x B=256 = 205M samples (4096 epochs)


In [17]:
# ============ eps sweep: find where the estimator actually works ============
import json, os, yaml, time

EPS_GRID    = [0.05, 0.02, 0.005, 0.001]
PROBE_STEPS = 250
PROBE_BATCH = 64          # ESS is a per-x0 average; small batch is fine and 4x faster

results = {}
for eps0 in EPS_GRID:
    p = json.loads(json.dumps(cfg))          # copy, don't mutate the real config
    p["anneal"]["eps0"]         = eps0
    p["anneal"]["eps_final"]    = eps0       # FREEZE eps so we isolate its effect
    p["anneal"]["anneal_steps"] = 10**9
    p["dataset"]["batch_size"]  = PROBE_BATCH
    p["train"]["total_steps"]   = PROBE_STEPS
    p["train"]["micro_batches"] = 1
    p["train"]["diag_every"]    = 0          # prox_multistart is 80 sequential fwd+bwd
    p["train"]["save_per_step"] = 10**9
    p["train"]["eval_per_step"] = 10**9
    p["eval"]["enabled"]        = False
    p["logging"] = {"use_wandb": False, "log_every_k": 10}   # jsonl only exists when W&B is off

    pcfg = f"{REPO}/configs/probe.yaml"
    yaml.safe_dump(p, open(pcfg, "w"), sort_keys=False)
    wd = f"{WORK}/runs/probe_eps{eps0}"
    get_ipython().system(f"rm -rf {wd}")

    print(f"\n=== eps0 = {eps0} ===", flush=True)
    t0 = time.time()
    rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29561 "
                   f"train.py --config configs/probe.yaml --workdir {wd} > {wd}.log 2>&1")
    rows = [json.loads(l) for l in open(f"{wd}/log/metrics.jsonl")] if rc == 0 else []
    ess  = [r["ess_frac"] for r in rows if "ess_frac" in r]
    if not ess:
        print("  FAILED - see", f"{wd}.log"); continue

    early, late = sum(ess[:3])/3, sum(ess[-3:])/3
    results[eps0] = {"early": early, "late": late, "min": min(ess),
                     "drift": late - early, "mins": (time.time()-t0)/60}
    print(f"  ESS  early {early:.2f} -> late {late:.2f}   (min {min(ess):.2f})"
          f"   [{results[eps0]['mins']:.1f} min]", flush=True)

# ---- verdict ----
print(f"\n{'eps0':>8} {'ESS early':>10} {'ESS late':>9} {'drift':>7}  verdict")
print("-" * 56)
ok = []
for e, r in sorted(results.items(), reverse=True):
    if   r["late"] >= 0.5 and r["drift"] > -0.15: v, good = "HEALTHY", True
    elif r["late"] >= 0.3:                        v, good = "marginal", False
    else:                                         v, good = "COLLAPSED", False
    if good: ok.append(e)
    print(f"{e:>8} {r['early']:>10.2f} {r['late']:>9.2f} {r['drift']:>+7.2f}  {v}")

if ok:
    EPS_START = max(ok)
    print(f"\n-> eps0 = {EPS_START}   (warmest that holds ESS>=0.5 and is not drifting down)")
    print(f"   suggested eps_final = {EPS_START/20:.1e}")
else:
    print("\n-> nothing held. eps is NOT the (only) problem - see note below.")


=== eps0 = 0.05 ===
  ESS  early 0.82 -> late 0.09   (min 0.09)   [4.9 min]

=== eps0 = 0.02 ===
  ESS  early 0.88 -> late 0.09   (min 0.09)   [5.1 min]

=== eps0 = 0.005 ===
  ESS  early 0.90 -> late 0.09   (min 0.09)   [5.1 min]

=== eps0 = 0.001 ===
  ESS  early 0.90 -> late 0.09   (min 0.09)   [5.1 min]

    eps0  ESS early  ESS late   drift  verdict
--------------------------------------------------------
    0.05       0.82      0.09   -0.73  COLLAPSED
    0.02       0.88      0.09   -0.79  COLLAPSED
   0.005       0.90      0.09   -0.81  COLLAPSED
   0.001       0.90      0.09   -0.81  COLLAPSED

-> nothing held. eps is NOT the (only) problem - see note below.


In [18]:
# ===== which mixture component holds the importance weight? =====
import math, torch, json
from ptflow.core.estimator import sample_proposal, log_gauss_iso
from ptflow.models.adapters import make_gen_fn, make_u_fn
from ptflow.models.builder import build_networks
from ptflow.data.factory import build_data
from ptflow.utils.misc import load_config

CK   = f"{WORK}/runs/probe_eps0.02/checkpoints/state_00000250.pt"
PCFG = load_config(f"{REPO}/configs/probe.yaml")
dev  = "cuda"

pot, gen, fs = build_networks(PCFG)
pay = torch.load(CK, map_location="cpu", weights_only=False)
pot.load_state_dict(pay["potential"]); gen.load_state_dict(pay["generator"])
pot, gen = pot.to(dev).eval(), gen.to(dev).eval()
u_fn, gen_fn = make_u_fn(pot, fs), make_gen_fn(gen, fs)

PCFG.dataset["batch_size"] = 64; PCFG.dataset["kwargs"]["num_workers"] = 0
b = build_data(PCFG); bd = b.preprocess_fn(next(iter(b.train_loader)))
x1 = fs.to_flat(bd["images"].to(dev).float())[:32]
c  = bd["labels"].to(dev).long()[:32]

B, K, eps = 32, 32, 0.02
torch.manual_seed(0)
x0 = torch.randn(B, fs.dim, device=dev)
with torch.no_grad():
    m, s = gen_fn(x0, c, 0.0)
    prop = sample_proposal(m, s, x0, eps=eps, K=K, alpha_def=0.1,
                           beta_data=0.1, x_anchor=x1[torch.randperm(B, device=dev)])
    u_y  = u_fn(prop.y.reshape(K*B, -1), c.repeat(K)).reshape(K, B)
    lw   = log_gauss_iso(prop.y - x0[None], 2*eps) - u_y - prop.log_q
    w    = torch.softmax(lw, dim=0)                      # (K, B)

kt, kn, kd = prop.counts
print(f"components: {kt} tilted | {kn} naive | {kd} data-anchored\n")
print("weight mass held by each block:")
for name, sl in (("tilted", slice(0, kt)), ("naive", slice(kt, kt+kn)),
                 ("data-anchored", slice(kt+kn, K))):
    print(f"  {name:15s} {float(w[sl].sum(0).mean()):6.1%}")

def ess(x):
    a = torch.logsumexp(x, 0); bq = torch.logsumexp(2*x, 0)
    return float(torch.exp(2*a - bq - math.log(x.shape[0])).mean())
print(f"\nESS/K overall      {ess(lw):.3f}")
print(f"ESS/K tilted only  {ess(lw[:kt]):.3f}   <- 'is the tilt itself high-variance?'")

print(f"\ns_eta   mean {float(s.mean()):+.2f}  min {float(s.min()):+.2f}  "
      f"max {float(s.max()):+.2f}   (0 = naive width; -6 = clamp floor)")
with torch.no_grad():
    print(f"u(data) {float(u_fn(x1, c).mean()):+10.1f}   "
          f"u(prox) {float(u_fn(m, c).mean()):+10.1f}   "
          f"u(noise) {float(u_fn(x0, c).mean()):+10.1f}")
print(f"out_scale {float(pot.out_scale):+.3f}   "
      f"displacement/dim {float(((m-x0).pow(2).sum(-1)/fs.dim).mean().sqrt()):.3f}")

components: 26 tilted | 3 naive | 3 data-anchored

weight mass held by each block:
  tilted            0.0%
  naive           100.0%
  data-anchored     0.0%

ESS/K overall      0.091
ESS/K tilted only  0.087   <- 'is the tilt itself high-variance?'

s_eta   mean -0.99  min -1.22  max -0.74   (0 = naive width; -6 = clamp floor)
u(data)      +30.9   u(prox)      +30.9   u(noise)      +31.0
out_scale +1.001   displacement/dim 0.001


/tmp/ipykernel_58/1762767732.py:54: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"out_scale {float(pot.out_scale):+.3f}   "


## 4 — Sanity (20 steps, tiny model)

End-to-end gate on both GPUs before committing hours.


In [ ]:
smoke = json.loads(json.dumps(load_config(f"{REPO}/configs/cifar10_unet_smoke.yaml")))
smoke["dataset"]["num_classes"] = 10
smoke["dataset"]["batch_size"] = 32
smoke.setdefault("eval", {})["enabled"] = False
smoke["logging"] = {"use_wandb": False, "log_every_k": 5}
yaml.safe_dump(smoke, open(f"{REPO}/configs/cifar10_pro_smoke.yaml", "w"), sort_keys=False)

get_ipython().system(f"rm -rf {WORK}/runs/smoke")
rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29551 "
               f"train.py --config configs/cifar10_pro_smoke.yaml --workdir {WORK}/runs/smoke")
assert rc == 0, "sanity run failed - read the traceback before spending GPU hours"
print(f"\nSANITY OK on {NGPU} GPU(s)")


## 5 — Timing probe → size this session, and get the ETA to target

Runs the **real** model briefly and reads the per-step times the trainer logs. The probe writes to a
throwaway workdir with W&B off (the local `metrics.jsonl` only exists when W&B is disabled).


In [ ]:
PROBE_STEPS = 60
probe_dir = f"{WORK}/runs/probe"
get_ipython().system(f"rm -rf {probe_dir}")

write_config(PROBE_STEPS, use_wandb=False)          # probe: local jsonl, no W&B pollution
rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29553 "
               f"train.py --config configs/cifar10_pro.yaml --workdir {probe_dir}")
assert rc == 0, "timing probe failed"

rows = [json.loads(l) for l in open(f"{probe_dir}/log/metrics.jsonl")]
times = [r["step_time"] for r in rows if "step_time" in r]
warm = times[len(times)//2:] or times
SEC_PER_IT = sorted(warm)[len(warm)//2]

# where does this session start from?
prev = sorted(glob.glob("/kaggle/input/*/runs/ptflow_cifar10/checkpoints/state_*.pt")
              + glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"))
RESUME_STEP = int(os.path.basename(prev[-1]).split("_")[1].split(".")[0]) if prev else 0

session_steps = int(SESSION_HOURS * 3600 / SEC_PER_IT)
STOP_AT = min(TARGET_STEPS, RESUME_STEP + session_steps)
write_config(STOP_AT, use_wandb=USE_WANDB)

samples_s = BATCH_SIZE / SEC_PER_IT
print(f"measured {SEC_PER_IT:.3f} s/it at B={BATCH_SIZE} on {NGPU} GPU(s)"
      f"   = {samples_s:.0f} samples/s = {samples_s*3600/1e6:.2f}M samples/hour")
print(f"effective throughput: {samples_s*PER_SAMPLE_GFLOP/1e3:.1f} TFLOPS\n")

print(f"resume from step : {RESUME_STEP:,}")
print(f"this session     : +{STOP_AT-RESUME_STEP:,} steps -> stop at {STOP_AT:,}")
print(f"target           : {TARGET_STEPS:,}  ({100*RESUME_STEP/TARGET_STEPS:.1f}% done now, "
      f"{100*STOP_AT/TARGET_STEPS:.1f}% after this session)")

remaining = TARGET_STEPS - STOP_AT
if remaining > 0:
    sess_left = remaining / max(1, session_steps)
    print(f"\nremaining        : {remaining:,} steps = {sess_left:.0f} more sessions")
    print(f"  1 person  (~2.5 sessions/wk): {sess_left/2.5:.0f} weeks")
    print(f"  7 people  (~17 sessions/wk) : {sess_left/17:.0f} weeks")
    print(f"  with fp16 AMP (~2.5x)       : {sess_left/17/2.5:.1f} weeks with 7 people")
else:
    print("\nTARGET REACHED - this session finishes the schedule.")


## 6 — Train this leg

Resumes from the newest checkpoint found in `/kaggle/input/**` or the workdir, restoring model, EMA,
optimizer **and anneal state**, and appends to the same W&B run.

**Relay handoff, for the next person:**
1. When this notebook finishes, **Save Version** so `/kaggle/working` is preserved.
2. Share that version's output as a dataset (public, or with the group).
3. The next person attaches it as a **Data source**, keeps `TARGET_STEPS`, `BATCH_SIZE`,
   `K_PROPOSALS`, `GEN_CH`, `WANDB_RUN_ID` **identical**, and runs from the top.

Two numbers certify the run: **`ess`** ≥ 0.3 (below that the trainer pauses the anneal and doubles
the generator:potential ratio on its own), and **`L_theta_phi_units`** converging. Raw `L_theta`
grows like 1/ε by design — that is not divergence.


In [16]:
os.makedirs(f"{WORKDIR}/checkpoints", exist_ok=True)
inp = sorted(glob.glob("/kaggle/input/*/runs/ptflow_cifar10/checkpoints/state_*.pt"))
if inp and not glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"):
    for p in inp[-2:]:
        shutil.copy(p, f"{WORKDIR}/checkpoints/{os.path.basename(p)}")
    print(f"resuming from {os.path.basename(inp[-1])}")
elif glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"):
    print("checkpoints already in the workdir; trainer resumes from the latest")
else:
    print("fresh run (step 0)")

_t0 = time.time()
rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29550 "
               f"train.py --config configs/cifar10_pro.yaml --workdir {WORKDIR}")
_h = (time.time() - _t0) / 3600
print(f"\ntraining wall time: {_h:.2f} h")
if rc != 0:
    print("WARNING: trainer exited non-zero (session limit? OOM?). Checkpoints are on disk; "
          "evaluation below uses the newest.")

ck = sorted(glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"))
DONE_STEP = int(os.path.basename(ck[-1]).split("_")[1].split(".")[0]) if ck else RESUME_STEP
print(f"progress: {DONE_STEP:,} / {TARGET_STEPS:,}  ({100*DONE_STEP/TARGET_STEPS:.1f}%)")
get_ipython().system(f"ls -lh {WORKDIR}/checkpoints/")


fresh run (step 0)


  0%|          | 0/800000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performance.
grad.sizes() = [192, 192, 1, 1], strides() = [192, 1, 192, 192]
bucket_view.sizes() = [192, 192, 1, 1], strides() = [192, 1, 1, 1] (Triggered internally at /pytorch/torch/csrc/distributed/c10d/reducer.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair perfor


training wall time: 0.20 h
progress: 0 / 800,000  (0.0%)
total 0


## 7 — Evaluation

Skipped on intermediate relay legs when `RUN_EVAL = False` — on a mid-relay session it is usually
better to spend the 2 hours training. Run it on the first session (to confirm the pipeline), then
every few legs to track progress, and always on the final one.


In [ ]:
from IPython.display import Image as IPImage, display

ckpts = sorted(glob.glob(f"{WORKDIR}/checkpoints/state_*.pt"))
assert ckpts, "no checkpoints - did training run?"
CKPT = ckpts[-1]
print("evaluating", os.path.basename(CKPT), "| RUN_EVAL =", RUN_EVAL)

CIFAR_NAMES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
if RUN_EVAL:
    for w in (0.0, 0.5):
        out = f"{WORK}/samples_w{w}.png"
        get_ipython().system(
            f'cd {REPO} && python inference.py sample --ckpt {CKPT} '
            f'--config configs/cifar10_pro.yaml --mode A --w {w} '
            f'--class-ids "0,1,2,3,4,5,6,7,8,9" --num-rows 8 --seed 42 --save-path {out}')
        print(f"Mode A (1-NFE), w={w}   columns: {', '.join(CIFAR_NAMES)}")
        display(IPImage(out))


### 7a — FID / IS: guidance sweep, then the headline number

Sweep `w` cheaply to find the best guidance weight, then re-run **only** the winner at 50k. A 50k
run per `w` would cost hours for no extra information.


In [ ]:
import pandas as pd

def run_eval(w, n, mode="A", extra=""):
    jout = f"{WORK}/eval_{mode}_w{w}_n{n}.json"
    rc = os.system(f"cd {REPO} && torchrun --nproc_per_node={NGPU} --master_port=29552 "
                   f"inference.py evaluate --ckpt {CKPT} --config configs/cifar10_pro.yaml "
                   f"--mode {mode} --w {w} --num-samples {n} --gen-bsz 250 {extra} "
                   f"--json-out {jout}")
    return json.load(open(jout)) if (rc == 0 and os.path.exists(jout)) else None

sweep, BEST_W, final = {}, 0.0, None
if RUN_EVAL:
    for w in (0.0, 0.25, 0.5, 1.0, 1.5):
        r = run_eval(w, FID_SWEEP_N)
        if r:
            sweep[w] = r
            print(f"  w={w:<5} FID {r.get('fid', float('nan')):8.2f}   IS {r.get('isc_mean', 0):6.2f}")
    assert sweep, "every sweep evaluation failed"
    BEST_W = min(sweep, key=lambda k: sweep[k].get("fid", float("inf")))
    print(f"\nbest w = {BEST_W} (FID {sweep[BEST_W]['fid']:.2f} at {FID_SWEEP_N} samples)")
    print(f"headline run: {FID_FINAL_N} samples ...")
    final = run_eval(BEST_W, FID_FINAL_N)
    if final:
        print(f"\n  FID {final['fid']:.2f}   IS {final.get('isc_mean',0):.2f}"
              f" +/- {final.get('isc_std',0):.2f}   (1-NFE, w={BEST_W}, {FID_FINAL_N} samples)")


### 7b — Mode B: the quality / compute dial

`n` damped fixed-point steps on the prox residual, starting from the Mode-A output. No retraining.
FID should fall monotonically with `n` — that curve is the evidence the generator really is
approximating the prox rather than an unrelated map.


In [ ]:
import matplotlib.pyplot as plt

mode_b = {}
if RUN_EVAL:
    for n in (0, 1, 2, 4):
        r = sweep.get(BEST_W) if n == 0 else run_eval(
            BEST_W, FID_SWEEP_N, mode="B", extra=f"--refine-steps {n} --gamma 0.5")
        if r:
            mode_b[n] = r.get("fid")
            print(f"  n={n}  NFE={n+1}  FID {mode_b[n]:8.2f}")
    if len(mode_b) > 1:
        ks = sorted(mode_b)
        plt.figure(figsize=(6, 4))
        plt.plot([k+1 for k in ks], [mode_b[k] for k in ks], "o-")
        plt.xlabel("NFE (1 = Mode A)"); plt.ylabel("FID"); plt.grid(alpha=0.3)
        plt.title(f"Mode-B refinement (w={BEST_W})"); plt.show()


### 7c — Exact normalized likelihood (w = 0)

The thing no other one-step model reports. Each value is an IWAE-style **bound**, so the honest
report is a ladder — NLL should decrease as `K_eval` grows, and flatten. Valid at `w = 0` only.


In [ ]:
if RUN_EVAL:
    LL = f"{WORK}/likelihood.json"
    rc = os.system(f"cd {REPO} && python inference.py likelihood --ckpt {CKPT} "
                   f"--config configs/cifar10_pro.yaml --num-samples 512 --batch-size 32 "
                   f"--ladder 16,64,256,1024 --json-out {LL}")
    if rc == 0 and os.path.exists(LL):
        ll = json.load(open(LL))
        ks = sorted(int(k.split("K")[1]) for k in ll if k.startswith("nll_K"))
        print(f"  {'K_eval':>8}  {'NLL (nats)':>12}  {'bits/dim':>10}")
        for k in ks:
            print(f"  {k:>8}  {ll[f'nll_K{k}']:12.2f}  {ll[f'nll_K{k}']/(3072*0.6931):10.4f}")
        plt.figure(figsize=(6, 4))
        plt.semilogx(ks, [ll[f"nll_K{k}"] for k in ks], "o-", base=2)
        plt.xlabel("K_eval"); plt.ylabel("NLL (nats)"); plt.grid(alpha=0.3)
        plt.title("IWAE ladder - should tighten monotonically"); plt.show()


## 8 — Session summary + handoff

In [ ]:
summary = {
    "run_id": WANDB_RUN_ID,
    "step": DONE_STEP, "target": TARGET_STEPS,
    "progress_pct": round(100*DONE_STEP/TARGET_STEPS, 2),
    "samples_seen_M": round(DONE_STEP*BATCH_SIZE/1e6, 1),
    "epochs": round(DONE_STEP*BATCH_SIZE/50000, 1),
    "batch_size": BATCH_SIZE, "K": K_PROPOSALS, "gen_ch": GEN_CH,
    "sec_per_it": round(SEC_PER_IT, 3),
}
if final:
    summary.update(FID_1NFE=round(final["fid"], 2),
                   IS=round(final.get("isc_mean", 0), 2), best_w=BEST_W)
if mode_b:
    summary["mode_b_fid_by_nfe"] = {k+1: (round(v, 2) if v else None) for k, v in mode_b.items()}
pd.DataFrame([summary]).T.to_csv(f"{WORK}/session_summary.csv", header=["value"])
print(json.dumps(summary, indent=2))

left = TARGET_STEPS - DONE_STEP
print(f"""
--------------------------------------------------------------------
HANDOFF -- next session

  1. Save Version (so /kaggle/working is preserved)
  2. Share that output as a dataset with the next person
  3. They attach it as a Data source and run this notebook unchanged

  Keep IDENTICAL: TARGET_STEPS={TARGET_STEPS}, BATCH_SIZE={BATCH_SIZE},
                  K_PROPOSALS={K_PROPOSALS}, GEN_CH={GEN_CH},
                  WANDB_RUN_ID='{WANDB_RUN_ID}'
  Change freely : SESSION_HOURS, RUN_EVAL

  progress {DONE_STEP:,}/{TARGET_STEPS:,} ({summary['progress_pct']}%), {left:,} steps left
--------------------------------------------------------------------""")
